# Results — 640 vs 1280 vs tiling, both models

Result tables of the study ([docs/proposal.md](../docs/proposal.md)). This notebook does
**no inference**: it only merges the CSVs written by
[highres.ipynb](highres.ipynb) (640 and 1280 full-image) and
[tiling.ipynb](tiling.ipynb) (640×640 native-resolution tiles, 20 % overlap + overlap
sweep) for one `EVAL_STAGE`, so every row comes from the same checkpoints and the same
shared evaluator ([`src/evaluation/metrics.py`](../src/evaluation/metrics.py)).

Tables:

1. **Overall** — mAP50–95, mAP50, recall and runtime per model × condition, with the
   change vs the 640 baseline (absolute, in AP points, and relative);
2. **Per object size** — disjoint groups tiny (<16²), small (16²–32²), medium
   (32²–96²), large (≥96² px): AP50–95, AP50, recall and object counts, with the change
   vs 640;
3. **Model capacity** — yolo26x − yolo26m inside each condition;
4. **Tile-overlap sensitivity** (yolo26m, 0 / 20 / 40 %) — a sensitivity analysis, not
   a selection: the main condition was fixed at 20 % before any result was examined.

All scores are on VisDrone2019-DET **test-dev**; validation was used only for checkpoint
selection. Numbers follow our protocol and are not official leaderboard scores. Scores
are shown ×100 (AP / recall points); deltas are in points unless marked ×.

## 1. Setup and inputs

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import DISJOINT_SIZE_GROUPS

SPLIT = "test"
EVAL_STAGE = os.environ.get("EVAL_STAGE", "finetune")
RUNS_DIR = ROOT / "runs"
HIGHRES_DIR = RUNS_DIR / "highres" / EVAL_STAGE
TILING_DIR = RUNS_DIR / "tiling" / EVAL_STAGE
RESULTS_DIR = RUNS_DIR / "results" / EVAL_STAGE
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "full-640": HIGHRES_DIR / f"highres_scores_{SPLIT}_640.csv",
    "full-1280": HIGHRES_DIR / f"highres_scores_{SPLIT}_1280.csv",
    "tiled-ov20": TILING_DIR / f"tiling_scores_{SPLIT}_ov20.csv",
}
SWEEP_CSV = TILING_DIR / f"tiling_overlap_sweep_{SPLIT}_yolo26m.csv"
CONDITIONS = list(INPUTS)
REFERENCE = "full-640"
MODELS = ["yolo26m", "yolo26x"]
GROUP_LABEL = {
    "tiny": "tiny (<16²)",
    "small_only": "small (16²-32²)",
    "medium": "medium (32²-96²)",
    "large": "large (≥96²)",
}

missing = [str(path) for path in INPUTS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"run highres.ipynb and tiling.ipynb with EVAL_STAGE={EVAL_STAGE} first: {missing}"
    )

frames = []
for condition, path in INPUTS.items():
    frame = pd.read_csv(path)
    frame["inference"] = condition
    frames.append(frame)
scores = pd.concat(frames, ignore_index=True).set_index(["model", "inference"]).sort_index()
weights = scores["weights"].groupby(level="model").unique()
print(f"stage={EVAL_STAGE}  ->  {RESULTS_DIR}")
for model, paths in weights.items():
    status = "OK" if len(paths) == 1 else "MISMATCH: conditions use different checkpoints!"
    print(f"{model}: {list(paths)}  [{status}]")

In [ ]:
def to_markdown(table: pd.DataFrame, digits: int = 2) -> str:
    """Plain GitHub markdown table (no tabulate dependency)."""
    table = table.reset_index()
    header = "| " + " | ".join(str(c) for c in table.columns) + " |"
    numeric = [pd.api.types.is_numeric_dtype(table[c]) for c in table.columns]
    rule = "|" + "|".join("---:" if is_num else "---" for is_num in numeric) + "|"

    def fmt(value: object) -> str:
        if isinstance(value, float):
            return "—" if pd.isna(value) else f"{value:.{digits}f}"
        return str(value)

    body = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in table.itertuples(index=False)]
    return "\n".join([header, rule, *body])


markdown_parts: list[str] = [f"# Result tables — stage `{EVAL_STAGE}`, {SPLIT}-dev\n"]


def publish(title: str, table: pd.DataFrame, name: str, digits: int = 2) -> None:
    """Show a table, save it as CSV and collect it for the markdown export."""
    table.to_csv(RESULTS_DIR / f"{name}.csv")
    display(Markdown(f"**{title}**"))
    display(table.round(digits))
    markdown_parts.append(f"## {title}\n\n{to_markdown(table, digits)}\n")

## 2. Table 1 — overall metrics and runtime

Runtime is per original image (batch 1, warm-up, CUDA sync; tiling counts all crops and
the merge). `Δ` = condition − 640 in AP / recall points; `×` = condition / 640.

In [ ]:
OVERALL = {
    "map50_95": "mAP50-95",
    "map50": "mAP50",
    "recall50_25": "Recall",
}
RUNTIME = {"runtime_mean_ms": "ms/img (mean)", "runtime_p95_ms": "ms/img (p95)"}

rows = []
for model in MODELS:
    ref = scores.loc[(model, REFERENCE)]
    for condition in CONDITIONS:
        row = scores.loc[(model, condition)]
        out = {"model": model, "condition": condition}
        out.update({label: 100 * row[key] for key, label in OVERALL.items()})
        out.update({label: row[key] for key, label in RUNTIME.items()})
        if condition != REFERENCE:
            out.update(
                {f"Δ {label}": 100 * (row[key] - ref[key]) for key, label in OVERALL.items()}
            )
            out["x runtime"] = row["runtime_mean_ms"] / ref["runtime_mean_ms"]
        rows.append(out)
table1 = pd.DataFrame(rows).set_index(["model", "condition"])
publish("Table 1 — overall (x100) and runtime", table1, "table1_overall")

## 3. Table 2 — per object size (disjoint groups)

Each object belongs to exactly one group by its area in original-image pixels. AP50 is
reported next to AP50–95 because a 1–2 px localisation error already pushes a tiny box
below IoU 0.75: AP50 says whether the object was *found*, AP50–95 how *precisely*.
`× AP` = relative change of AP50–95 vs 640 — read it together with `Δ`, since small
baselines make ratios look large.

In [ ]:
size_rows = []
for model in MODELS:
    ref = scores.loc[(model, REFERENCE)]
    for group in DISJOINT_SIZE_GROUPS:
        for condition in CONDITIONS:
            row = scores.loc[(model, condition)]
            out = {
                "model": model,
                "size": GROUP_LABEL[group],
                "condition": condition,
                "objects": int(row[f"n_{group}"]),
                "AP50-95": 100 * row[f"ap_{group}"],
                "AP50": 100 * row[f"ap50_{group}"],
                "Recall": 100 * row[f"recall_{group}"],
            }
            if condition != REFERENCE:
                out["Δ AP50-95"] = 100 * (row[f"ap_{group}"] - ref[f"ap_{group}"])
                out["Δ AP50"] = 100 * (row[f"ap50_{group}"] - ref[f"ap50_{group}"])
                out["Δ Recall"] = 100 * (row[f"recall_{group}"] - ref[f"recall_{group}"])
                base_ap = ref[f"ap_{group}"]
                out["x AP"] = row[f"ap_{group}"] / base_ap if base_ap > 0 else float("nan")
            size_rows.append(out)
table2 = pd.DataFrame(size_rows).set_index(["model", "size", "condition"])
for model in MODELS:
    publish(f"Table 2 — per object size, {model} (x100)", table2.loc[model], f"table2_size_{model}")

counts = scores.loc[(MODELS[0], REFERENCE), [f"n_{g}" for g in DISJOINT_SIZE_GROUPS]]
print(
    "objects per group:",
    {GROUP_LABEL[g]: int(counts[f"n_{g}"]) for g in DISJOINT_SIZE_GROUPS},
    "| total:",
    int(scores.loc[(MODELS[0], REFERENCE), "n_all"]),
)

In [ ]:
# Δ AP50-95 and Δ recall vs 640 per size group: where each condition helps or hurts.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)
groups = [GROUP_LABEL[g] for g in DISJOINT_SIZE_GROUPS]
bar_conditions = [c for c in CONDITIONS if c != REFERENCE]
width = 0.2
for ax, metric in zip(axes, ["Δ AP50-95", "Δ Recall"]):
    for i, (model, condition) in enumerate([(m, c) for m in MODELS for c in bar_conditions]):
        values = [table2.loc[(model, g, condition), metric] for g in groups]
        positions = [k + (i - 1.5) * width for k in range(len(groups))]
        ax.bar(positions, values, width, label=f"{model} {condition}")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(groups)), groups)
    ax.set_ylabel(f"{metric} vs 640 (points)")
    ax.set_title(f"{metric} vs 640 per object size")
    ax.grid(alpha=0.3, axis="y")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "delta_by_size.png", dpi=150)
plt.show()

## 4. Table 3 — model capacity: yolo26x − yolo26m per condition

The secondary question: does added capacity help small objects as much as added input
resolution? Compare these deltas with the condition deltas in Tables 1–2.

In [ ]:
CAPACITY = {
    "map50_95": "mAP50-95",
    "map50": "mAP50",
    "recall50_25": "Recall",
    **{f"ap_{g}": f"AP50-95 {g}" for g in DISJOINT_SIZE_GROUPS},
    **{f"recall_{g}": f"Recall {g}" for g in DISJOINT_SIZE_GROUPS},
}
cap_rows = []
for condition in CONDITIONS:
    m_row = scores.loc[("yolo26m", condition)]
    x_row = scores.loc[("yolo26x", condition)]
    out = {"condition": condition}
    out.update({f"Δ {label}": 100 * (x_row[k] - m_row[k]) for k, label in CAPACITY.items()})
    out["x runtime (x/m)"] = x_row["runtime_mean_ms"] / m_row["runtime_mean_ms"]
    cap_rows.append(out)
table3 = pd.DataFrame(cap_rows).set_index("condition")
publish("Table 3 — yolo26x - yolo26m (points)", table3.T, "table3_capacity")

## 5. Table 4 — tile-overlap sensitivity (yolo26m)

Sensitivity analysis only: the main tiling condition (20 %) was fixed in the proposal,
and the sweep is reported on test-dev without selecting the best value from it.

In [ ]:
if SWEEP_CSV.exists():
    sweep = pd.read_csv(SWEEP_CSV).sort_values("overlap")
    sweep_cols = {
        "map50_95": "mAP50-95",
        "recall50_25": "Recall",
        **{f"ap_{g}": f"AP50-95 {g}" for g in DISJOINT_SIZE_GROUPS if f"ap_{g}" in sweep},
        "recall_tiny": "Recall tiny",
    }
    table4 = pd.DataFrame({label: 100 * sweep[key] for key, label in sweep_cols.items()}).assign(
        **{
            "ms/img (mean)": sweep["runtime_mean_ms"].to_numpy(),
            "ms/img (p95)": sweep["runtime_p95_ms"].to_numpy(),
        }
    )
    table4.index = [f"{o:.0%}" for o in sweep["overlap"]]
    table4.index.name = "overlap"
    publish("Table 4 — tile overlap sweep, yolo26m (x100)", table4, "table4_overlap")
else:
    print(f"overlap sweep not found at {SWEEP_CSV}")

## 6. Accuracy–runtime trade-off

In [ ]:
colors = {"yolo26m": "tab:blue", "yolo26x": "tab:orange"}
markers = {"full-640": "o", "full-1280": "s", "tiled-ov20": "^"}
panels = [
    ("map50_95", "mAP50-95"),
    ("ap50_tiny", "AP50 tiny (<16² px)"),
    ("recall_tiny", "Recall tiny (<16² px)"),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for model in MODELS:
    rows = scores.loc[model].reindex(CONDITIONS).sort_values("runtime_mean_ms")
    for ax, (metric, title) in zip(axes, panels):
        ax.plot(rows["runtime_mean_ms"], 100 * rows[metric], "-", color=colors[model], alpha=0.4)
        for condition, row in rows.iterrows():
            ax.scatter(
                row["runtime_mean_ms"],
                100 * row[metric],
                marker=markers[condition],
                color=colors[model],
                s=70,
                label=f"{model} {condition}",
            )
        ax.set_title(f"{title} vs runtime")
for ax in axes:
    ax.set_xlabel("runtime mean (ms / original image)")
    ax.set_ylabel("score x100")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "tradeoff.png", dpi=150)
plt.show()

In [ ]:
markdown_path = RESULTS_DIR / "results_tables.md"
markdown_path.write_text("\n".join(markdown_parts), encoding="utf-8")
print(f"saved tables -> {RESULTS_DIR} (CSV + {markdown_path.name}, PNG figures)")